# Практична робота №2: побудова та оцінювання коректного ML-експерименту

**Варіант:** 7 — Bike Sharing Dataset (той самий набір, що в ПР1 і ЛР1).
**Ціль:** `cnt` — погодинна кількість оренд велосипедів.

Схема: постановка задачі → train/test split → baseline → кандидатні моделі → 5-fold часова CV → порівняння → вибір → одноразовий test → аналіз коректності.

In [1]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.bike_workflow import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    chronological_split,
    load_bike_data,
    make_preprocessor,
    split_features_target,
)

WORK_DIR = Path.cwd().parent
FIGURES = WORK_DIR / "figures"
RESULTS = WORK_DIR / "results"
FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

## 1. Постановка задачі та гіпотеза

- **Об'єкт спостереження:** одна година роботи системи Capital Bikeshare (2011–2012).
- **Вхідні ознаки:** календар (`season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`) і погода (`weathersit`, `temp`, `atemp`, `hum`, `windspeed`).
- **Ціль:** `cnt` — кількість оренд за годину; практичний зміст — очікуване навантаження на систему для планування парку велосипедів.
- **Основна гіпотеза:** регресійні ML-моделі дають суттєво менший RMSE, ніж константний baseline, і ця перевага зберігається на пізнішому, не баченому періоді.
- **Основна метрика:** RMSE. Вона вимірюється в орендах за годину і сильніше штрафує великі промахи у години пік, які найдорожчі для планування. MAE і R² — додаткові.

## 2. Формування train/test

Дані впорядковано за датою та годиною. Перші 80% — train, останні 20% — test. `random_state` тут не потрібен: поділ детермінований і хронологічний, бо задача — прогноз майбутнього (умова $t_{train} < t_{test}$ з методички).

In [2]:
df = load_bike_data()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = chronological_split(X, y)

split_info = pd.DataFrame(
    {
        "Частина": ["train", "test"],
        "Рядків": [len(X_train), len(X_test)],
        "Від": [df.loc[X_train.index, "dteday"].min().date(), df.loc[X_test.index, "dteday"].min().date()],
        "До": [df.loc[X_train.index, "dteday"].max().date(), df.loc[X_test.index, "dteday"].max().date()],
        "Середнє cnt": [y_train.mean(), y_test.mean()],
    }
)
display(split_info.style.format({"Середнє cnt": "{:.1f}"}))
print("Ознаки моделі:", list(X.columns))
print("Витокові поля відсутні:", {"casual", "registered", "instant", "dteday"}.isdisjoint(X.columns))

,Частина,Рядків,Від,До,Середнє cnt
0,train,13903,2011-01-01,2012-08-07,174.6
1,test,3476,2012-08-07,2012-12-31,248.8


Ознаки моделі: ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
Витокові поля відсутні: True


Test-період (вересень–грудень 2012) має вищий середній попит (248,8 проти 174,6 у train): система зростала. Тому test свідомо складніший за train, і частину розриву CV↔test слід очікувати заздалегідь.

## 3. Перевірка preprocessing

Використовується preprocessing із ПР1: медіанна імпутація + `StandardScaler` для `yr, temp, atemp, hum, windspeed`; імпутація модою + `OneHotEncoder(handle_unknown="ignore")` для семи категоріальних полів. Усі кроки знаходяться в `ColumnTransformer`, який входить у `Pipeline` кожної моделі, тому `fit` відбувається лише на train-частині поточного fold.

In [3]:
check = make_preprocessor().fit(X_train)
scaler = check.named_transformers_["numeric"].named_steps["scaler"]
print("Ознак після preprocessing:", len(check.get_feature_names_out()))
print("Середні scaler збігаються з train:", np.allclose(scaler.mean_, X_train[NUMERIC_FEATURES].mean().to_numpy()))
print("Середні scaler НЕ дорівнюють середнім усього набору:", not np.allclose(scaler.mean_, X[NUMERIC_FEATURES].mean().to_numpy()))

Ознак після preprocessing: 60
Середні scaler збігаються з train: True
Середні scaler НЕ дорівнюють середнім усього набору: True


## 4–6. Baseline, кандидатні моделі та часова крос-валідація

Методичка пропонує `KFold(shuffle=True)`, але для цього набору він некоректний: fold-и з перемішаних годин дозволяють навчатися на майбутньому і перевірятися на минулому. Тому використано `TimeSeriesSplit(n_splits=5)`: validation завжди пізніший за train. Однакові fold-и використовуються для всіх моделей (це вимога однакових умов порівняння); у розділі 10 показано, наскільки перемішаний KFold завищує якість.

Кандидати (гіперпараметри фіксовані до запуску, без підбору):

| Модель | Чому включена |
|---|---|
| `DummyRegressor(mean)` | основний baseline $\hat y = \bar y_{train}$ |
| `DummyRegressor(median)` | альтернативний baseline для правоасиметричної цілі |
| `LinearRegression` | найпростіша адитивна модель на one-hot ознаках |
| `Ridge(alpha=1)` | та сама модель з L2-штрафом; перевірка, чи стабілізує регуляризація temp/atemp |
| `DecisionTreeRegressor(min_samples_leaf=10)` | нелінійна модель, що сама будує взаємодії `hr × workingday` |
| `RandomForestRegressor(150 дерев, min_samples_leaf=3, max_features=0.5)` | ансамбль дерев, який зменшує дисперсію окремого дерева |

In [4]:
cv = TimeSeriesSplit(n_splits=5)
folds = pd.DataFrame(
    [
        {"fold": i, "train_rows": len(tr), "valid_rows": len(va),
         "train_до": df.loc[X_train.index[tr[-1]], "dteday"].date(),
         "valid_від": df.loc[X_train.index[va[0]], "dteday"].date(),
         "valid_до": df.loc[X_train.index[va[-1]], "dteday"].date()}
        for i, (tr, va) in enumerate(cv.split(X_train), 1)
    ]
)
display(folds)

,fold,train_rows,valid_rows,train_до,valid_від,valid_до
0,1,2318,2317,2011-04-11,2011-04-11,2011-07-17
1,2,4635,2317,2011-07-17,2011-07-17,2011-10-22
2,3,6952,2317,2011-10-22,2011-10-22,2012-01-27
3,4,9269,2317,2012-01-27,2012-01-27,2012-05-02
4,5,11586,2317,2012-05-02,2012-05-02,2012-08-07


In [5]:
def make_pipeline(model):
    return Pipeline([("preprocessor", make_preprocessor()), ("model", model)])


models = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "Baseline (median)": DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=1)": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(min_samples_leaf=10, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, min_samples_leaf=3, max_features=0.5, n_jobs=-1, random_state=RANDOM_STATE
    ),
}
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}

summary_rows, fold_rows = [], []
for name, model in models.items():
    start = time.perf_counter()
    scores = cross_validate(make_pipeline(model), X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    elapsed = time.perf_counter() - start
    rmse = -scores["test_rmse"]
    for i, value in enumerate(rmse, 1):
        fold_rows.append({"Модель": name, "fold": i, "RMSE": value})
    summary_rows.append(
        {
            "Модель": name,
            "CV RMSE": rmse.mean(),
            "Std RMSE": rmse.std(ddof=1),
            "CV MAE": -scores["test_mae"].mean(),
            "CV R²": scores["test_r2"].mean(),
            "Train RMSE": -scores["train_rmse"].mean(),
            "Час CV, с": elapsed,
        }
    )

cv_results = pd.DataFrame(summary_rows).sort_values("CV RMSE").reset_index(drop=True)
fold_results = pd.DataFrame(fold_rows)
cv_results.to_csv(RESULTS / "cv_results.csv", index=False)
fold_results.to_csv(RESULTS / "cv_fold_rmse.csv", index=False)
display(cv_results.style.format({c: "{:.3f}" for c in cv_results.columns if c != "Модель"}))

,Модель,CV RMSE,Std RMSE,CV MAE,CV R²,Train RMSE,"Час CV, с"
0,Random Forest,94.529,23.359,67.372,0.627,26.757,7.771
1,Decision Tree,107.061,22.338,74.065,0.539,41.675,0.390
2,Linear Regression,108.762,21.641,80.494,0.545,69.139,0.397
3,Ridge (alpha=1),108.833,21.714,80.582,0.545,69.141,0.251
4,Baseline (mean),180.481,47.759,138.079,-0.212,123.280,0.184
5,Baseline (median),192.989,54.069,144.239,-0.379,127.767,0.194


In [6]:
fold_table = fold_results.pivot(index="Модель", columns="fold", values="RMSE").loc[cv_results["Модель"]]
display(fold_table.style.format("{:.1f}"))

fold,1,2,3,4,5
Модель,,,,,
Random Forest,123.3,75.1,73.1,115.5,85.6
Decision Tree,132.9,90.4,78.7,122.2,111.2
Linear Regression,117.5,91.0,82.8,116.5,136.1
Ridge (alpha=1),117.6,91.1,82.6,116.6,136.2
Baseline (mean),183.9,155.0,119.5,196.4,247.6
Baseline (median),194.6,172.0,120.3,209.8,268.4


Стандартне відхилення обчислено з поправкою $k-1$ (`ddof=1`), як у формулі методички; тому воно трохи більше за значення `np.std`, наведені в ЛР1.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.1, 1]})
plot_data = cv_results.sort_values("CV RMSE", ascending=False)
colors = ["#c46a5a" if "Baseline" in n else "#3f7f93" for n in plot_data["Модель"]]
axes[0].barh(plot_data["Модель"], plot_data["CV RMSE"], xerr=plot_data["Std RMSE"], color=colors, capsize=4)
axes[0].set(title="5-fold часова CV: середній RMSE ± std", xlabel="CV RMSE, оренд/год (менше — краще)", ylabel="")
for i, (m, s) in enumerate(zip(plot_data["CV RMSE"], plot_data["Std RMSE"])):
    axes[0].text(m + s + 3, i, f"{m:.1f} ± {s:.1f}", va="center", fontsize=9)
axes[0].set_xlim(0, plot_data["CV RMSE"].max() + plot_data["Std RMSE"].max() + 45)
for name in cv_results["Модель"]:
    part = fold_results[fold_results["Модель"] == name]
    style = "--" if "Baseline" in name else "-"
    axes[1].plot(part["fold"], part["RMSE"], style, marker="o", label=name)
axes[1].set(title="RMSE на кожному часовому fold", xlabel="Fold (пізніший період →)", ylabel="RMSE", xticks=range(1, 6))
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "cv_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Аналіз результатів крос-валідації

In [8]:
base = cv_results.set_index("Модель").loc["Baseline (mean)", "CV RMSE"]
best_name = cv_results.loc[~cv_results["Модель"].str.contains("Baseline"), "Модель"].iloc[0]
analysis = cv_results.copy()
analysis["Покращення vs baseline, %"] = (1 - analysis["CV RMSE"] / base) * 100
display(analysis[["Модель", "CV RMSE", "Std RMSE", "Покращення vs baseline, %"]].style.format(
    {"CV RMSE": "{:.2f}", "Std RMSE": "{:.2f}", "Покращення vs baseline, %": "{:.1f}"}))

rf_folds = fold_table.loc["Random Forest"]
lin_folds = fold_table.loc["Linear Regression"]
tree_folds = fold_table.loc["Decision Tree"]
print("RF краще за Linear у fold-ах:", int((rf_folds < lin_folds).sum()), "з 5; середня парна різниця:",
      round((lin_folds - rf_folds).mean(), 2))
print("RF краще за Tree у fold-ах:", int((rf_folds < tree_folds).sum()), "з 5; середня парна різниця:",
      round((tree_folds - rf_folds).mean(), 2))

,Модель,CV RMSE,Std RMSE,"Покращення vs baseline, %"
0,Random Forest,94.53,23.36,47.6
1,Decision Tree,107.06,22.34,40.7
2,Linear Regression,108.76,21.64,39.7
3,Ridge (alpha=1),108.83,21.71,39.7
4,Baseline (mean),180.48,47.76,0.0
5,Baseline (median),192.99,54.07,-6.9


RF краще за Linear у fold-ах: 4 з 5; середня парна різниця: 14.23
RF краще за Tree у fold-ах: 5 з 5; середня парна різниця: 12.53


**Висновки з CV:**

- Усі чотири ML-моделі перевищили обидва baseline. Медіанний baseline гірший за середній (193,0 проти 180,5), бо за правої асиметрії медіана систематично занижує години пік, а RMSE їх сильно штрафує.
- `Linear Regression` і `Ridge(alpha=1)` практично тотожні (108,76 і 108,83): на ~13,9 тис. об'єктів і 60 ознаках L2-штраф з `alpha=1` майже не змінює коефіцієнти, тому регуляризація тут не дає ні виграшу, ні стабілізації.
- `Decision Tree` за середнім (107,06) близький до лінійних моделей, але має інший профіль помилок: сильно програє на fold 1 і виграє на fold 5. Тобто одне дерево нестабільне між періодами.
- `Random Forest` має найменший середній RMSE (94,53) і виграє у лінійної моделі в 4 з 5 fold-ів; у fold 1 (найменший train — лише 2 318 годин 2011 року) він гірший. Розкид між fold-ами у всіх моделей великий (≈ 21–23), бо різні fold-и — це різні сезони з різним рівнем попиту, а не випадкові підвибірки.
- Перевага RF над лінійною моделлю (≈ 14 RMSE, 13%) менша за std між fold-ами, тому окремо за середнім вона не є статистично переконливою; переконливішою її робить парне порівняння на тих самих fold-ах (4/5 перемог, середній виграш ≈ 14).

## 8. Вибір фінальної моделі

Рішення приймається до перегляду test. Критерії: середній CV RMSE, розкид, перевага над baseline, складність і витрати.

- **Random Forest** — найменший CV RMSE 94,53 ± 23,36, покращення над baseline ≈ 47,6%, перемога над Linear у 4/5 і над Decision Tree у 5/5 парних порівнянь.
- Витрати: CV RF займає секунди, а не частки секунди, як у лінійних моделях, — для погодинного прогнозу це неістотно.
- Інтерпретованість нижча, ніж у лінійної моделі, але задача (планування навантаження) не вимагає пояснення кожного коефіцієнта.

**Обрано: `Random Forest`.** Лінійна регресія залишається простою контрольною моделлю.

## 9. Фінальне оцінювання на test set (один раз)

In [9]:
final_model = make_pipeline(models[best_name]).fit(X_train, y_train)
baseline_model = make_pipeline(models["Baseline (mean)"]).fit(X_train, y_train)
prediction = final_model.predict(X_test)
baseline_prediction = baseline_model.predict(X_test)

test_metrics = pd.DataFrame(
    [
        {"Модель": "Baseline (mean)", "MAE": mean_absolute_error(y_test, baseline_prediction),
         "RMSE": root_mean_squared_error(y_test, baseline_prediction), "R²": r2_score(y_test, baseline_prediction)},
        {"Модель": best_name, "MAE": mean_absolute_error(y_test, prediction),
         "RMSE": root_mean_squared_error(y_test, prediction), "R²": r2_score(y_test, prediction)},
    ]
)
test_metrics.to_csv(RESULTS / "test_metrics.csv", index=False)
display(test_metrics.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R²": "{:.3f}"}))
cv_rmse = cv_results.set_index("Модель").loc[best_name, "CV RMSE"]
cv_std = cv_results.set_index("Модель").loc[best_name, "Std RMSE"]
test_rmse = test_metrics.loc[1, "RMSE"]
print(f"CV RMSE = {cv_rmse:.2f} ± {cv_std:.2f}; test RMSE = {test_rmse:.2f}; різниця = {test_rmse - cv_rmse:+.2f}")

,Модель,MAE,RMSE,R²
0,Baseline (mean),174.985,232.608,-0.113
1,Random Forest,61.628,91.636,0.827


CV RMSE = 94.53 ± 23.36; test RMSE = 91.64; різниця = -2.89


In [10]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, prediction, alpha=0.25, s=16, color="#3f7f93")
limit = max(y_test.max(), prediction.max()) * 1.02
ax.plot([0, limit], [0, limit], "--", color="#c44e52", label="ŷ = y (ідеальний прогноз)")
ax.set(xlim=(0, limit), ylim=(0, limit), title="Random Forest: фактичні та передбачені cnt (test)",
       xlabel="Фактичне cnt", ylabel="Передбачене cnt")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "actual_vs_predicted.png", dpi=160, bbox_inches="tight")
plt.show()

bins = pd.cut(y_test, [0, 100, 300, 500, 1000])
by_range = pd.DataFrame({"y": y_test, "pred": prediction, "bin": bins}).groupby("bin", observed=True).apply(
    lambda g: pd.Series({"n": len(g), "Середня похибка (ŷ−y)": (g["pred"] - g["y"]).mean(),
                         "RMSE": root_mean_squared_error(g["y"], g["pred"])}), include_groups=False)
display(by_range.style.format({"n": "{:.0f}", "Середня похибка (ŷ−y)": "{:+.1f}", "RMSE": "{:.1f}"}))

,n,Середня похибка (ŷ−y),RMSE
bin,,,
"(0, 100]",1115,+14.1,41.5
"(100, 300]",1194,-11.7,58.2
"(300, 500]",676,-87.1,106.7
"(500, 1000]",491,-155.9,177.8


Test RMSE близький до CV-оцінки і лежить у межах одного std між fold-ами, тобто CV не дала оптимістичної оцінки. Графік і таблиця за діапазонами показують систематичне **заниження** при високому попиті: для `cnt` 300–500 середня похибка −87, для `cnt > 500` — −156 оренд. Ліс усереднює значення листків і не може передбачити більше, ніж бачив у train, а test-період має вищий загальний рівень попиту (зростання системи в 2012 році). Для низького попиту прогнози, навпаки, трохи завищені.

## 10. Аналіз коректності експерименту

Щоб показати, що схема не формальна, ті самі моделі прогнали за **некоректними** схемами. Ці результати не використовуються для вибору моделі.

In [11]:
rf_pipe = make_pipeline(models["Random Forest"])
checks = []

shuffled = cross_validate(rf_pipe, X_train, y_train, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
                          scoring="neg_root_mean_squared_error")
checks.append({"Схема": "RF, KFold(shuffle=True) замість TimeSeriesSplit", "RMSE": -shuffled["test_score"].mean(),
               "Коректна оцінка": cv_rmse})

Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
random_split_rmse = root_mean_squared_error(yb, make_pipeline(models["Random Forest"]).fit(Xa, ya).predict(Xb))
checks.append({"Схема": "RF, випадковий train_test_split замість хронологічного", "RMSE": random_split_rmse,
               "Коректна оцінка": test_rmse})

X_leak = df.drop(columns=["cnt", "instant", "dteday"])
X_leak_train = X_leak.loc[X_train.index]
leak_pre = make_preprocessor()
leak_pre.transformers.append(("leak", StandardScaler(), ["casual", "registered"]))
leak = cross_validate(Pipeline([("preprocessor", leak_pre), ("model", LinearRegression())]), X_leak_train, y_train,
                      cv=cv, scoring="neg_root_mean_squared_error")
checks.append({"Схема": "Linear, + casual і registered (target leakage)", "RMSE": -leak["test_score"].mean(),
               "Коректна оцінка": cv_results.set_index("Модель").loc["Linear Regression", "CV RMSE"]})

checks = pd.DataFrame(checks)
checks["Завищення якості, %"] = (1 - checks["RMSE"] / checks["Коректна оцінка"]) * 100
checks.to_csv(RESULTS / "leakage_checks.csv", index=False)
display(checks.style.format({"RMSE": "{:.3f}", "Коректна оцінка": "{:.3f}", "Завищення якості, %": "{:.1f}"}))

,Схема,RMSE,Коректна оцінка,"Завищення якості, %"
0,"RF, KFold(shuffle=True) замість TimeSeriesSplit",51.322,94.529,45.7
1,"RF, випадковий train_test_split замість хронологічного",51.065,91.636,44.3
2,"Linear, + casual і registered (target leakage)",0.000,108.762,100.0


In [12]:
fig, ax = plt.subplots(figsize=(10, 4.5))
labels = ["RF: shuffled KFold\nvs TimeSeriesSplit", "RF: випадковий split\nvs хронологічний", "Linear: + casual/registered\nvs без них"]
xpos = np.arange(len(labels))
ax.bar(xpos - 0.2, checks["RMSE"], 0.4, label="Некоректна схема", color="#c46a5a")
ax.bar(xpos + 0.2, checks["Коректна оцінка"], 0.4, label="Коректна схема", color="#3f7f93")
for x, a, b in zip(xpos, checks["RMSE"], checks["Коректна оцінка"]):
    ax.text(x - 0.2, a + 2, f"{a:.1f}", ha="center", fontsize=9)
    ax.text(x + 0.2, b + 2, f"{b:.1f}", ha="center", fontsize=9)
ax.set_xticks(xpos, labels)
ax.set(title="Як некоректна схема занижує RMSE", ylabel="RMSE, оренд/год")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "leakage_checks.png", dpi=160, bbox_inches="tight")
plt.show()

- Перемішаний KFold дає RMSE ≈ 51 замість ≈ 95: модель «бачить» сусідні години того самого дня з майбутнього, що відсутнє у реальному прогнозі.
- Випадковий split дає test RMSE ≈ 51 замість ≈ 92 на хронологічному тесті — та сама причина.
- Додавання `casual` і `registered` дає RMSE ≈ 0: `cnt = casual + registered`, лінійна модель просто відновлює суму. Це не якість, а витік цілі.

**Чек-лист коректності:**

| Питання | Відповідь |
|---|---|
| Test відокремлено до вибору моделі? | Так, хронологічно, до будь-якого `fit` |
| Test використовувався під час налаштування? | Ні; гіперпараметри фіксовані, вибір — за CV на train |
| Preprocessing навчався лише на train? | Так, усередині `Pipeline` на кожному fold (перевірено середні scaler) |
| Однакові folds для всіх моделей? | Так, один об'єкт `TimeSeriesSplit(5)` |
| Baseline оцінено за тією ж схемою? | Так, `DummyRegressor` у тому ж `Pipeline` і на тих самих fold-ах |
| Метрики у звіті = виводу програми? | Так, таблиці збережено в `results/*.csv` |

Обмеження: test-період уже використовувався в ЛР1 для оцінки іншої моделі, тому для ПР2 він не є абсолютно «свіжим». Модель ПР2 обрано лише за CV, test переглянуто один раз.

## 11. Висновки

1. Перевірялася гіпотеза, що регресійні моделі суттєво зменшують RMSE відносно константного прогнозу на пізнішому періоді.
2. Baseline (середнє train) має CV RMSE 180,48 ± 47,76 і test RMSE 232,61 (R² < 0).
3. Порівнювались Linear Regression, Ridge, Decision Tree і Random Forest на однакових 5 часових fold-ах з однаковим preprocessing.
4. Найменший середній CV RMSE має Random Forest: 94,53 ± 23,36.
5. Розкид між fold-ами великий у всіх моделей, бо fold-и — різні сезони; RF стабільно кращий за лінійну модель у 4 з 5 fold-ів.
6. Відносно baseline RF зменшив CV RMSE приблизно на 47,6%, test RMSE — з 232,6 до 91,6.
7. На test: MAE = 61,63, RMSE = 91,64, R² = 0,827.
8. Test RMSE (91,6) нижчий за CV-середнє (94,5) і в межах одного std — оцінки узгоджуються.
9. Гіпотеза підтвердилась. Додатково показано, що перемішаний KFold і випадковий split занижують RMSE майже вдвічі, а `casual/registered` дають фіктивний RMSE ≈ 0.
10. Інженерне рішення: Random Forest придатний як модель для агрегованого планування, але систематично недооцінює піки зростаючого попиту; у ЛР1 на тих самих fold-ах HistGradientBoosting мав CV RMSE 77,3, тому для продакшн-прогнозу він залишається кращим вибором, а RF — сильнішою альтернативою лінійним моделям.